# Step 1: Get Features From Multiple Datasets
- Using [pybiber](https://pypi.org/project/pybiber/)

In [1]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

from transformers import logging
logging.set_verbosity_error()

In [2]:
import pybiber as pb

import polars as pl
import os
import numpy as np
import random
import torch
# from scipy.stats import zscore
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import gc
import importlib.resources as resources

from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, matthews_corrcoef
from scipy.stats import pearsonr, spearmanr

In [3]:
DEVICE = 0 if torch.cuda.is_available() else -1
FILE_PATH = 'getText/datasetsPrep'
OUTPUT_DIR = 'biberOutputs'
SAMPLE_SIZE = 2
batch_idx = 0

In [4]:
ZERO_SHOT_MODELS = [
    "cross-encoder/nli-deberta-v3-small", # low capacity
    "typeform/distilbert-base-uncased-mnli", # medium capacity
    "valhalla/distilbart-mnli-12-3", # higher capacity
]

In [5]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [6]:
# Load all data.
list_of_dfs = []
for folder in os.listdir(f'./{FILE_PATH}'):
    if os.path.isdir(f'./{FILE_PATH}/{folder}'):
        for file in os.listdir(f'./{FILE_PATH}/{folder}'):
            if file.endswith(".csv"):
                temp_file_path = f'./{FILE_PATH}/{folder}/{file}'
                temp_tag = file.replace('.csv', '')
                temp_df = pl.read_csv(temp_file_path)
                temp_df = (
                    temp_df
                    .with_row_index("index_num") # , offset=1) if you want to start index from 1
                    .with_columns(
                        (pl.lit(temp_tag) + "_" + pl.col("index_num").cast(pl.Utf8)).alias("doc_id")
                    )).select(['text', 'doc_id'])
                list_of_dfs.append(temp_df)

combined = pl.concat(list_of_dfs, how="vertical")
assert combined.select(pl.col("doc_id").n_unique()).item() == combined.height, "There should be no duplicates in the dataset."

In [7]:
# Remove invalid data.
temp_df = combined.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(temp_df.group_by("tag").len())
print(f"Total Texts Before Empty String Removal: {len(temp_df)}")

temp_df = combined.with_columns(pl.col("text").str.strip_chars().alias("text")).filter(pl.col("text").is_not_null() & (pl.col("text") != ""))

temp_df = temp_df.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(temp_df.group_by("tag").len())
print(f"Total Texts After Empty String Removal: {len(temp_df)}")

shape: (14, 2)
┌────────────────────────────────┬────────┐
│ tag                            ┆ len    │
│ ---                            ┆ ---    │
│ str                            ┆ u32    │
╞════════════════════════════════╪════════╡
│ 20NewsGroups                   ┆ 18846  │
│ trumpTweets                    ┆ 56571  │
│ simSUM                         ┆ 10000  │
│ bbcNews                        ┆ 2225   │
│ clinicalDialogueSummarizations ┆ 3603   │
│ atis                           ┆ 4978   │
│ huffPostNews                   ┆ 209527 │
│ medicalAbstracts               ┆ 14438  │
│ clinc150                       ┆ 23700  │
│ yahoo                          ┆ 87362  │
│ banking77                      ┆ 13069  │
│ augmentedClinicalNotes         ┆ 30000  │
│ dementiaAudio                  ┆ 549    │
│ syntheticCareHomeNurseNotes    ┆ 5783   │
└────────────────────────────────┴────────┘
Total Texts Before Empty String Removal: 480651
shape: (14, 2)
┌────────────────────────────────┬────────

In [8]:
# Randomly sample from dataframe.
temp_df = temp_df.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
temp_df = temp_df.to_pandas()
temp_df = (temp_df.groupby("tag")).apply(lambda x: x.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE + batch_idx)) ; batch_idx += 1 # Increase batch_idx every time to ensure new random samples.
temp_df = pl.from_pandas(temp_df)


In [9]:
# Custom biber implementation without category dependence. 
def custom_mda_biber(features_df, threshold: float = 0.35):

        """Project results onto Biber's dimensions.

        Parameters
        ----------
        threshold:
            The factor loading threshold (in absolute value)
            used to calculate dimension scores.

        """
        # Load packaged Biber promax loadings
        try:
            with resources.as_file(
                resources.files("pybiber.data").joinpath("biber_loadings.csv")
            ) as p:
                loadings_df = pl.read_csv(str(p))
        except Exception as e:
            raise FileNotFoundError(
                "Could not load 'biber_loadings.csv' from pybiber.data"
            ) from e

        # Identify factor columns and intersecting features
        factor_cols = [
            c for c in loadings_df.columns if c.startswith("factor_") and not c.endswith("7")  # Remove factor 7 as it is not used or defined in Biber's original 6 dimensions.
        ]
        if not factor_cols:
            raise ValueError("Biber loadings file has no factor_* columns.")
        n_factors = len(factor_cols)

        # Align features present in the user matrix and in the loadings
        user_feats = set(features_df.columns)
        common_feats = [f for f in loadings_df.get_column("feature").to_list() if f in user_feats]
        if not common_feats:
            raise ValueError(
                "No overlapping features between data and Biber loadings."
            )

        # Trim to common features (preserve loading order)
        m_trim = features_df.select(common_feats)
        L = (
            loadings_df
            .filter(pl.col("feature").is_in(common_feats))
            .select(factor_cols)
            .to_numpy()
        )

        # Standardize counts (z-score per feature) using safe routine
        x = m_trim.to_numpy()
        m_z, zero_var_idx = pb.biber_analyzer._safe_standardize(x, ddof=1)
        if zero_var_idx:
            print(
                "Zero-variance features retained (neutral scaling) in "
                "projection: %s",
                [m_trim.columns[i] for i in zero_var_idx],
            )

        # Thresholded sum/difference per Biber MDA convention
        pos = (L > threshold).T  # shape: (k, p)
        neg = (L < -threshold).T

        dim_scores = []
        for i in range(n_factors):
            pos_sum = (
                np.sum(m_z[:, pos[i]], axis=1)
                if pos[i].any() else np.zeros(m_z.shape[0])
            )
            neg_sum = (
                np.sum(m_z[:, neg[i]], axis=1)
                if neg[i].any() else np.zeros(m_z.shape[0])
            )
            scores = pos_sum - neg_sum
            dim_scores.append(scores)

        dim_scores = pl.from_numpy(
            np.array(dim_scores).T,
            schema=["factor_" + str(i) for i in range(1, n_factors + 1)],
        )


        # Loadings returned for the actually used features (aligned)
        loadings = (loadings_df.filter(pl.col("feature").is_in(common_feats)).select(["feature", *factor_cols]))

        # Assign results
        return dim_scores, loadings
        

In [10]:
# Set up df for use.
df = pl.DataFrame({
    "doc_id": temp_df['doc_id'].to_list(),
    "text": temp_df['text'].to_list()
})

# Light preprocessing to strip extra whitespace.
df = df.with_columns(
    pl.col("text")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .str.replace_all(r"^\s*-\s*", "") # Remove dashes at the beginning of texts.
    .str.replace_all(r"^\s*\d+\.\s*", "") # Remove numbers in 1., 2., 3. format at the beginning of the text. 
)

pybiber_pipeline = pb.PybiberPipeline(model="en_core_web_sm")
features, tokens = pybiber_pipeline.run(df, return_tokens=True, normalize=True)
features = features.with_columns(pl.col("doc_id").str.split("_").list.get(0).alias("category"))
# Full feature list can be found here: https://browndw.github.io/pybiber/feature-categories.html
print(f" -------- Features-------- ")
print(features)

# Multi-Dimensional Analysis - see https://browndw.github.io/pybiber/biber-analyzer.html#comparison-with-bibers-original-dimensions for factor mapping
# Explanation of the factor mapping to dimensions can be found here: https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html
'''
Factor 1: Involved vs. Informational Production (negative to positive)
Factor 2: Narrative vs. Non-narrative Concerns (negative to positive)
Factor 3: Explicit vs. Situation-dependent Reference (negative to positive)
Factor 4: Overt Expression of Persuasion (negative to positive)
Factor 5: Abstract vs. Non-abstract Information (negative to positive)
Factor 6: On-line Informational Elaboration (negative to positive)
'''

dim_scores, loadings = custom_mda_biber(features)
print(f" -------- MDA Loadings -------- ")
print(loadings)
print(f" -------- MDA Dimension Scores -------- ")
print(dim_scores)

[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length


 -------- Features-------- 
shape: (28, 69)
┌─────────┬─────────┬─────────┬─────────┬─────────┬───┬────────┬────────┬────────┬────────┬────────┐
│ doc_id  ┆ f_01_pa ┆ f_02_pe ┆ f_03_pr ┆ f_04_pl ┆ … ┆ f_64_p ┆ f_65_c ┆ f_66_n ┆ f_67_n ┆ catego │
│ ---     ┆ st_tens ┆ rfect_a ┆ esent_t ┆ ace_adv ┆   ┆ hrasal ┆ lausal ┆ eg_syn ┆ eg_ana ┆ ry     │
│ str     ┆ e       ┆ spect   ┆ ense    ┆ erbials ┆   ┆ _coord ┆ _coord ┆ thetic ┆ lytic  ┆ ---    │
│         ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆   ┆ inatio ┆ inatio ┆ ---    ┆ ---    ┆ str    │
│         ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆   ┆ n      ┆ n      ┆ f64    ┆ f64    ┆        │
│         ┆         ┆         ┆         ┆         ┆   ┆ ---    ┆ ---    ┆        ┆        ┆        │
│         ┆         ┆         ┆         ┆         ┆   ┆ f64    ┆ f64    ┆        ┆        ┆        │
╞═════════╪═════════╪═════════╪═════════╪═════════╪═══╪════════╪════════╪════════╪════════╪════════╡
│ 20NewsG ┆ 21.3675 ┆ 4.27350 ┆ 68.3760 ┆ 0.0  

In [11]:
os.makedirs(f"./{OUTPUT_DIR}/", exist_ok=True)

def flatten_for_csv(df):
    # Work on a copy to avoid modifying original.
    df_flat = df.clone()
    for c, dtype in zip(df_flat.columns, df_flat.dtypes):
        if dtype == pl.List:
            # Join list elements into string with commas.
            df_flat = df_flat.with_columns(
                pl.Series(df_flat[c].name, [",".join(map(str, x)) if x is not None else "" for x in df_flat[c]])
            )
        elif dtype == pl.Struct:
            # Convert struct to string representation.
            df_flat = df_flat.with_columns(
                pl.Series(df_flat[c].name, df_flat[c].cast(pl.Utf8))
            )
    return df_flat

# Get Z-Scores from Biber analysis.
biber_dimensions = (dim_scores).to_pandas()
# Get factor columns.
factor_cols = [c for c in biber_dimensions.columns if c.startswith("factor")]
# biber_dimensions[factor_cols] = biber_dimensions[factor_cols].apply(zscore)
for c in factor_cols:
    biber_dimensions[f"{c}_label"] = biber_dimensions[c] > 0
biber_dimensions = pl.from_pandas(biber_dimensions)
print(biber_dimensions)

shape: (28, 12)
┌─────────┬─────────┬─────────┬─────────┬─────────┬───┬────────┬────────┬────────┬────────┬────────┐
│ factor_ ┆ factor_ ┆ factor_ ┆ factor_ ┆ factor_ ┆ … ┆ factor ┆ factor ┆ factor ┆ factor ┆ factor │
│ 1       ┆ 2       ┆ 3       ┆ 4       ┆ 5       ┆   ┆ _2_lab ┆ _3_lab ┆ _4_lab ┆ _5_lab ┆ _6_lab │
│ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆   ┆ el     ┆ el     ┆ el     ┆ el     ┆ el     │
│ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆   ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│         ┆         ┆         ┆         ┆         ┆   ┆ bool   ┆ bool   ┆ bool   ┆ bool   ┆ bool   │
╞═════════╪═════════╪═════════╪═════════╪═════════╪═══╪════════╪════════╪════════╪════════╪════════╡
│ 15.0092 ┆ -0.7803 ┆ -0.4739 ┆ 1.09778 ┆ 0.61421 ┆ … ┆ false  ┆ false  ┆ true   ┆ true   ┆ true   │
│ 43      ┆ 03      ┆ 62      ┆ 2       ┆ 2       ┆   ┆        ┆        ┆        ┆        ┆        │
│ 8.72005 ┆ -0.6686 ┆ -1.1593 ┆ 2.88632 ┆ 2.67857 ┆ … ┆ false  ┆ false  ┆ t

In [12]:
# Write CSV files.
flatten_for_csv(biber_dimensions).write_csv(f"./{OUTPUT_DIR}/mda_dim_scores{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)
flatten_for_csv(loadings).write_csv(f"./{OUTPUT_DIR}/mda_loadings{RANDOM_STATE + batch_idx - 1}.csv", float_precision=15)

# Write JSON files.
loadings.write_json(f"./{OUTPUT_DIR}/mda_loadings{RANDOM_STATE + batch_idx - 1}.json")
biber_dimensions.write_json(f"./{OUTPUT_DIR}/mda_dim_scores{RANDOM_STATE + batch_idx - 1}.json")

In [13]:
# Exact mapping taken from https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html which has extracted the same from Biber and Conrad's Variation in English (https://doi.org/10.4324/9781315840888)
BIBER_LABEL_MAP = {
    "factor_1": {"informational": -1, "involved": 1},
    "factor_2": {"non-narrative": -1, "narrative": 1},
    "factor_3": {"situation-dependent": -1, "explicit": 1},
    "factor_4": {"non-persuasive": -1, "persuasive": 1},
    "factor_5": {"non-abstract": -1, "abstract": 1},
    "factor_6": {"compressed": -1, "elaborated": 1}
}

# Using different prompt templates increases robustness. 
TEMPLATES = [
    "This text is {}.",
    "This text is written in a {} style.",
    "The writing style of this text is {}.",
    "This text can be described as {}."
]

In [14]:
temp_df = temp_df.to_pandas()
texts = temp_df['text'].values.tolist()
doc_ids = temp_df['doc_id'].values.tolist()

In [15]:
assert len(texts) == len(doc_ids), "texts and doc_ids are not of the same length."

In [16]:
rows = []
for model_name in ZERO_SHOT_MODELS:
    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=DEVICE
    )
    for i in range(len(texts)):
        temp_factors = {'model_name': model_name, 'doc_id': doc_ids[i]}
        for factor, description in BIBER_LABEL_MAP.items():
            final_dimension_score = 0
            for template in TEMPLATES:
                outputs = classifier(
                    texts[i],
                    candidate_labels=list(description.keys()),
                    hypothesis_template=template,
                    batch_size=8,
                    multi_label = True
                )
                for label, score in zip(outputs['labels'], outputs['scores']):
                    final_dimension_score += description[label] * score
            final_dimension_score = final_dimension_score / len(TEMPLATES)
            temp_factors[factor] = final_dimension_score
        rows.append(temp_factors)
    # Free memory.
    del classifier
    torch.cuda.empty_cache()
    gc.collect()

    # break

df = pd.DataFrame(rows)


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [17]:
dimension_labels_verbose = {
    "factor_1_label": "informational_vs_involved",
    "factor_2_label": "non-narrative_vs_narrative",
    "factor_3_label": "situation-dependent_vs_explicit",
    "factor_4_label": "non-persuasive_vs_persuasive",
    "factor_5_label": "non-abstract_vs_abstract",
    "factor_6_label": "compressed_vs_elaborated"
}

# Weighted Averages
# model_weights = {
#     "cross-encoder/nli-deberta-v3-small": 0.25,
#     "typeform/distilbert-base-uncased-mnli": 0.25,
#     "valhalla/distilbart-mnli-12-3": 0.50
# }

# weighted_rows = []
# for model_name, group in df.groupby("model_name"):
#     weight = model_weights.get(model_name, 1.0)  # default 1 if not in dict
#     weighted_group = group.copy()
#     for factor in ["factor_1", "factor_2", "factor_3", "factor_4", "factor_5", "factor_6"]:
#         weighted_group[factor] = weighted_group[factor] * weight
#     weighted_rows.append(weighted_group)

# weighted_df = pd.concat(weighted_rows)
# mean_scores = weighted_df.drop(columns='model_name').groupby("doc_id").sum().reset_index()

# Simple averaging will prevent over-confidence. 
mean_scores = df.drop(columns='model_name').groupby("doc_id").mean().reset_index()


# # Get factor columns.
factor_cols = [c for c in mean_scores.columns if c.startswith("factor")]
# # Get Z-Scores from zero-shot analysis.
# mean_scores[factor_cols] = mean_scores[factor_cols].apply(zscore)
for c in factor_cols:
    mean_scores[f"{c}_label"] = mean_scores[c] > 0
mean_scores = pl.from_pandas(mean_scores)

results_cont = {}
for f in ["factor_1", "factor_2", "factor_3", "factor_4", "factor_5", "factor_6"]:
    pearson = pearsonr(biber_dimensions[f], mean_scores[f])[0]
    spearman = spearmanr(biber_dimensions[f], mean_scores[f])[0]
    mse = mean_squared_error(biber_dimensions[f], mean_scores[f])
    rmse = root_mean_squared_error(biber_dimensions[f], mean_scores[f])
    mae = mean_absolute_error(biber_dimensions[f], mean_scores[f])
    results_cont[f] = {"pearson": pearson, "spearman": spearman, "MSE": mse, "RMSE": rmse, "MAE": mae}
continuous_df = pd.DataFrame(results_cont).T
continuous_df = continuous_df.rename(index={
    "factor_1": "informational_vs_involved",
    "factor_2": "non-narrative_vs_narrative",
    "factor_3": "situation-dependent_vs_explicit",
    "factor_4": "non-persuasive_vs_persuasive",
    "factor_5": "non-abstract_vs_abstract",
    "factor_6": "compressed_vs_elaborated"
})
continuous_df['dimension'] = continuous_df.index
print(pl.from_pandas((continuous_df)))

results_bin = {}
for f in ["factor_1_label", "factor_2_label", "factor_3_label", "factor_4_label", "factor_5_label", "factor_6_label"]:
    acc = accuracy_score(biber_dimensions[f], mean_scores[f])
    prec = precision_score(biber_dimensions[f], mean_scores[f])
    rec = recall_score(biber_dimensions[f], mean_scores[f])
    f1 = f1_score(biber_dimensions[f], mean_scores[f])
    kappa = cohen_kappa_score(biber_dimensions[f], mean_scores[f])
    mcc = matthews_corrcoef(biber_dimensions[f], mean_scores[f])
    results_bin[f] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "kappa": kappa, "MCC": mcc}

classification_df = pd.DataFrame(results_bin).T
classification_df = classification_df.rename(index={
    "factor_1_label": "informational_vs_involved",
    "factor_2_label": "non-narrative_vs_narrative",
    "factor_3_label": "situation-dependent_vs_explicit",
    "factor_4_label": "non-persuasive_vs_persuasive",
    "factor_5_label": "non-abstract_vs_abstract",
    "factor_6_label": "compressed_vs_elaborated"
})
classification_df['dimension'] = classification_df.index
print(pl.from_pandas((classification_df)))

shape: (6, 6)
┌───────────┬───────────┬───────────┬──────────┬──────────┬─────────────────────────────────┐
│ pearson   ┆ spearman  ┆ MSE       ┆ RMSE     ┆ MAE      ┆ dimension                       │
│ ---       ┆ ---       ┆ ---       ┆ ---      ┆ ---      ┆ ---                             │
│ f64       ┆ f64       ┆ f64       ┆ f64      ┆ f64      ┆ str                             │
╞═══════════╪═══════════╪═══════════╪══════════╪══════════╪═════════════════════════════════╡
│ -0.083675 ┆ 0.013136  ┆ 77.442647 ┆ 8.80015  ┆ 7.71766  ┆ informational_vs_involved       │
│ 0.162044  ┆ 0.122125  ┆ 7.760877  ┆ 2.785835 ┆ 2.09892  ┆ non-narrative_vs_narrative      │
│ -0.068109 ┆ -0.137346 ┆ 4.079986  ┆ 2.019897 ┆ 1.68579  ┆ situation-dependent_vs_explici… │
│ 0.432998  ┆ 0.270278  ┆ 7.213522  ┆ 2.6858   ┆ 2.261691 ┆ non-persuasive_vs_persuasive    │
│ 0.417376  ┆ 0.569559  ┆ 2.757841  ┆ 1.660675 ┆ 1.202117 ┆ non-abstract_vs_abstract        │
│ -0.252348 ┆ -0.414019 ┆ 6.919478  ┆ 2.63049 

/home/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
